In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.func import vmap, grad, functional_call, jacrev, vjp
import wandb
from accelerate.test_utils.testing import get_backend
from typing import Optional
from core.models import NoisyMLP, load_model_from_artifact
from core.callbacks import WandBCallback
from lightning.pytorch.loggers import WandbLogger
from lightning import Trainer
from core.data import MNISTDataModule
from lightning.pytorch.callbacks import EarlyStopping
device, n_devices, _ = get_backend()

torch.set_float32_matmul_precision("highest")

## Warmup: Fitting gradients of a known function class

In [3]:
# Define your RFunction model (unchanged)
class RFunction(nn.Module):
    def __init__(self):
        super().__init__()
        self.theta = nn.Parameter(torch.tensor([1.0, 1.0]))  # Initialize parameters

    def forward(self, A):
        return torch.sum(self.theta[0] * torch.exp(self.theta[1] * A))  # Example function R(A, θ)

# For this example, we'll use the same simulated data, but in batches
A_data = torch.randn(500, 5)  # More data points
true_theta = torch.tensor([2.0, 0.5]) 
true_gradients = true_theta[0] * true_theta[1] * torch.exp(true_theta[1] * A_data)  # Simulated ∇R(A_i)
# Make sure to reshape true_gradients to match the batch size

# Create dataset and dataloader
dataset = TensorDataset(A_data, true_gradients)
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Instantiate model
model = RFunction()
optimizer = optim.Adam(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

# Training loop with batches
num_epochs = 30

for epoch in range(num_epochs):
    epoch_loss = 0.0
    batch_count = 0
    
    for A_batch, true_gradients_batch in dataloader:
        optimizer.zero_grad()
        
        # Make inputs require gradients
        A_batch = A_batch.detach().requires_grad_()
        
        # Compute predicted gradient ∇R(A, θ) via autograd
        R_values = model(A_batch)
        gradients = torch.autograd.grad(R_values, A_batch, create_graph=True)[0]
        
        # Compute loss for this batch

        loss = loss_fn(gradients, true_gradients_batch)
        
        # Backpropagate and update θ
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        batch_count += 1
    
    avg_epoch_loss = epoch_loss / batch_count
    print(f"Epoch {epoch}: Average Loss = {avg_epoch_loss:.8f}")

# Learned parameters
print("Learned theta:", model.theta.detach().numpy())

Epoch 0: Average Loss = 1.03890332
Epoch 1: Average Loss = 0.67412865
Epoch 2: Average Loss = 0.11518243
Epoch 3: Average Loss = 0.05882251
Epoch 4: Average Loss = 0.03538274
Epoch 5: Average Loss = 0.02638052
Epoch 6: Average Loss = 0.01754166
Epoch 7: Average Loss = 0.01285180
Epoch 8: Average Loss = 0.00933581
Epoch 9: Average Loss = 0.00696030
Epoch 10: Average Loss = 0.00508136
Epoch 11: Average Loss = 0.00380863
Epoch 12: Average Loss = 0.00277562
Epoch 13: Average Loss = 0.00202402
Epoch 14: Average Loss = 0.00147649
Epoch 15: Average Loss = 0.00108000
Epoch 16: Average Loss = 0.00079202
Epoch 17: Average Loss = 0.00056398
Epoch 18: Average Loss = 0.00041308
Epoch 19: Average Loss = 0.00028414
Epoch 20: Average Loss = 0.00020811
Epoch 21: Average Loss = 0.00014020
Epoch 22: Average Loss = 0.00009737
Epoch 23: Average Loss = 0.00006703
Epoch 24: Average Loss = 0.00004683
Epoch 25: Average Loss = 0.00003142
Epoch 26: Average Loss = 0.00002130
Epoch 27: Average Loss = 0.00001423
Ep

## Linear Network with L1 / L2 penalty

In [2]:
from core.models import LinearNetwork
from core.callbacks import WandBCallback
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import WandbLogger
from lightning import Trainer
    
# Instantiate the model
input_dim = 10
output_dim = 1
hidden_dim = 200

model = LinearNetwork(input_dim, output_dim, hidden_dim, 
                      l1_lambda=1., l1_smooth=0.01,
                      l2_lambda=0.0, 
                      lr=1e-3)

# fake data
N = 5000
X = torch.randn(N, input_dim)
betas = 5*torch.randn(input_dim) 
y = (betas @ X.T).view(-1, 1) + 0.5 * torch.randn(N, 1)  # Linear relationship with noise
# Create a DataLoader for training and testing
dataset = TensorDataset(X, y)
# train test split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=3, drop_last=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=3, drop_last=True)


logger = WandbLogger(project="inductive-bias", name="linear-net")
trainer = Trainer(max_epochs=200, 
                  logger=logger, 
                  callbacks=[WandBCallback(), EarlyStopping(monitor="train/loss", patience=15, mode="min")], 
                  accelerator=device, 
                  devices=n_devices)
trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=test_dataloader)


/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jrudoler/.cache/pypoetry/virtualenvs/inductive ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


wandb: Currently logged in as: jhrudoler (jhrudoler-penn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



  | Name      | Type       | Params | Mode 
-------------------------------------------------
0 | linear    | Sequential | 2.4 K  | train
1 | loss_func | MSELoss    | 0      | train
-------------------------------------------------
2.4 K     Trainable params
0         Non-trainable params
2.4 K     Total params
0.010     Total estimated model params size (MB)
4         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

epoch,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▇▇▇▇▇▇▇▇█
train/loss,█▇▅▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇█████
val/loss,█▆▆▅▄▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,109
train/loss,14.26377
trainer/global_step,13749
val/loss,14.2867


### Using per-sample gradients to estimate bias

In [ ]:
from core.bias import RidgeBias, SmoothLassoBias, LassoBias, ElasticNet

# For this example, we'll use the same simulated data, but in batches
params = dict(model.named_parameters())
flattened_params = torch.cat([p.view(-1) for p in params.values()])
# buffers = dict(model.named_buffers())

# Define function that returns model output
def model_output(params, x):
    return functional_call(model, params, (x,))

# Create dataset and dataloader
dataset = TensorDataset(X, y)
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

# Instantiate model
# r_model = RidgeBias()
# wandb.init(project="inductive-bias", name="ridge-bias")
# r_model = SmoothLassoBias(smooth=0.1, alpha_init=0.5)
# wandb.init(project="inductive-bias", name="smooth-lasso-bias")
# r_model = LassoBias()
# wandb.init(project="inductive-bias", name="nonsmooth-lasso-bias")
r_model = ElasticNet()
wandb.init(project="inductive-bias", name="elastic-net")

wandb.watch(r_model)
optimizer = optim.Adam(r_model.parameters(), lr=1e-3)

# Training loop with batches
num_epochs = 200

for epoch in range(num_epochs):
    epoch_loss = 0.0
    batch_count = 0
    
    for X_batch, y_batch in dataloader:
        # Zero the gradients
        optimizer.zero_grad()
        flattened_params = flattened_params.detach().requires_grad_()

        # vector-Jacobian product, returns function (model_output) applied to primals (params, X) 
        # and a function that computes the vector-Jacobian product
        predictions, vjp_func = vjp(model_output, params, X_batch)
        # vjp with residuals, then select the derivative w.r.t. params
        # factor of 2 comes from gradient of the mse loss
        vjp_result = vjp_func(2*(y_batch - predictions))[0] # is this sign convention correct?
        true_gradients_batch = torch.cat([v.view(-1) for v in vjp_result.values()]) / batch_size

        # Compute predicted gradient ∇R(A, θ) via autograd
        R_val = r_model(flattened_params)
        gradients = torch.autograd.grad(R_val, flattened_params, create_graph=True)[0]
        # Compute loss for this batch
        loss = torch.nn.functional.mse_loss(
            gradients, 
            true_gradients_batch, 
            reduction='mean')
        
        # Backpropagate and update θ
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        batch_count += 1

        ## Early stopping

    
    avg_epoch_loss = epoch_loss / batch_count
    # print(f"Epoch {epoch}: Average Loss = {avg_epoch_loss:.8f}")
    wandb.log({"loss": avg_epoch_loss})
    # wandb.log({"learning_rate": optimizer.param_groups[0]['lr']})
    # Learned parameters
    for name, param in r_model.named_parameters():
        # print(f"Parameter {name}: {param.detach().numpy()}")
        wandb.log({name: param.detach().numpy()})

wandb.finish()

lambda_1,▁▂▆▇▇████████▇█▇███▇▇██▇███████▇██████▇█
lambda_2,█▆▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss,█▅▆▇▆▆▁▃▆▃▂▅▃▃▅▂▃▅▃▃▄▂▁▂▅▅▁▃▄▃▅▂▅▅▅▃▅▆▁▄
lambda_1,0.97872
lambda_2,0.00466
loss,0.0062


In [3]:
from core.estimators import BiasWithMSE, BiasWithAutodiffLoss 
from core.bias import ElasticNet
from importlib import reload
from core import estimators
reload(estimators)

dataset = TensorDataset(X, y)
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
bias_estimator = BiasWithMSE(
    predictive_model=model,
    # predictive_loss_fn=torch.nn.functional.mse_loss,
    bias_model=ElasticNet(smooth=0.01),
    grad_match_loss_fn=torch.nn.functional.mse_loss,
)
bias_trainer = Trainer(
    max_epochs=200,
    accelerator=device,
    devices=n_devices,
    callbacks=[
        WandBCallback(),
        EarlyStopping(monitor="train/loss", patience=15, mode="min"),
    ],
    logger=WandbLogger(project="inductive-bias", name="linear-ridge"),
)
bias_trainer.fit(bias_estimator, train_dataloaders=dataloader)

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'predictive_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['predictive_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'bias_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['bias_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so


  | Name             | Type          | Params | Mode 
-----------------------------------------------------------
0 | predictive_model | LinearNetwork | 2.4 K  | train
1 | bias_model       | ElasticNet    | 2      | train
-----------------------------------------------------------
2.4 K     Trainable params
0         Non-trainable params
2.4 K     Total params
0.010     Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

bias/lambda_1,▃▂▁▁▁▂▂▂▃▃▃▄▄▄▄▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇██████████
bias/lambda_2,█▇▆▅▅▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇█████
train/loss,▆▆▆▄▅▁▄▄▄▃▃▄▄▄▄▃▂▂▂▂▃▂▂▇▄▁▄▃▂▃▆▃▃█▃▃▃▃▄▂
trainer/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
bias/lambda_1,0.87794
bias/lambda_2,0.01739
epoch,22
train/loss,0.00463
trainer/global_step,3549


#### Testing vjp

In [ ]:
params = dict(model.named_parameters())
# buffers = dict(model.named_buffers())

# Define function that returns model output
def model_output(params, x):
    return functional_call(model, params, (x,))

# def single_output(params, x_single):
#     # Return a scalar prediction per example 
#     return functional_call(model, params, (x_single,)).squeeze()

# Compute Jacobian of output w.r.t. weights
# x = torch.randn(10, input_dim)
gradient_wrt_params = jacrev(model_output)(params, X[:10])
# grad_per_sample = vmap(grad(model_output), in_dims=(None, 0))(params, X)
predictions = model_output(params, X[:10])
product = (-(y[:10]-predictions)).T @ gradient_wrt_params['linear.0.weight'].view(10, -1)
product.view(100, -1)

In [ ]:
# Compute Jacobian of output w.r.t. weights
# x = torch.randn(10, input_dim)
# gradient_wrt_params = jacrev(model_output)(params, X[:10])
with torch.no_grad():
    # vector-Jacobian product, returns function (model_output) applied to primals (params, X) 
    # and a function that computes the vector-Jacobian product
    predictions, vjp_func = vjp(model_output, params, X[:10])
    # vjp with residuals, then select the derivative w.r.t. params
    vjp_result =  vjp_func(-(y[:10] - predictions))[0] 
print(vjp_result['linear.0.weight'])

## Dropout regularization in MNIST

Train a model without dropout

In [13]:
noiseless_model = NoisyMLP(dropout_rate=0.0, out_features=1)
wandb_logger = WandbLogger(project="inductive-bias", name="noiseless-mlp", log_model=True)
mnist = MNISTDataModule(batch_size=32)
trainer = Trainer(max_epochs=25,
                  logger=wandb_logger, 
                #   callbacks=[EarlyStopping(monitor="val/loss")]
                  )
trainer.fit(noiseless_model, mnist) 
wandb.finish()

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jrudoler/.cache/pypoetry/virtualenvs/inductive ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type              | Params | Mode 
--------------------------------------------------------
0 | loss_func | BCEWithLogitsLoss | 0      | train
1 | layers    | Sequential        | 150 K  | train
--------------------------------------------------------
150 K     Trainable params
0         Non-trainable params
150 K     Total params
0.601     Total estimated model params size (MB)
14        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=25` reached.


epoch,▁▁▁▁▁▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▇▇▇▇▇▇▇▇▇▇████
train/loss,█▄▂▁▆▁▁▁▁▂▂▁▁▁▁▂▁▁▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇█████
val/loss,▆▃▂▁▁▂▁▂▂▂▄▃▃▂▄▄▂▃▃▃█▄▄▅▃
epoch,24
train/loss,0.0
trainer/global_step,46874
val/loss,0.03689


Train a model with dropout, which we suspect is equivalent to some L2 regularization

In [ ]:
noisy_model = NoisyMLP(dropout_rate=0.3, out_features=1)  # Initialize the model with dropout
wandb_logger = WandbLogger(project="inductive-bias", name="noisy-mlp", log_model=True)
mnist = MNISTDataModule(batch_size=32)
trainer = Trainer(max_epochs=25,
                  logger=wandb_logger, 
                  callbacks=[EarlyStopping(monitor="val/loss")])
trainer.fit(noisy_model, mnist) 
wandb.finish()

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | loss_func | CrossEntropyLoss | 0      | train
1 | layers    | Sequential       | 150 K  | train
-------------------------------------------------------
150 K     Trainable params
0         Non-trainable params
150 K     Total params
0.601     Total estimated model params size (MB)
14        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/jrudoler/Library/Caches/pypoetry/virtualenvs/inductive-bias-XFXDbqEv-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


RuntimeError: 0D or 1D target tensor expected, multi-target not supported

In [2]:
noisy_model = load_model_from_artifact(NoisyMLP, 'jhrudoler-penn/inductive-bias/model-sxjn7tc3:latest')
noiseless_model = load_model_from_artifact(NoisyMLP, 'jhrudoler-penn/inductive-bias/model-n7awb9ov:latest')
no_stopping_model = load_model_from_artifact(NoisyMLP, 'jhrudoler-penn/inductive-bias/model-yv15t2cq:v0')
mnist = MNISTDataModule(batch_size=32)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  


In [18]:
torch.nn.functional.softmax(noisy_model(torch.randn(2, 1, 28, 28)))

/var/folders/7h/662tdm8d6sn0krrht717wzmm0000gq/T/ipykernel_82092/550448832.py:1: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  torch.nn.functional.softmax(noisy_model(torch.randn(2, 1, 28, 28)))


tensor([[1.],
        [1.]], grad_fn=<SoftmaxBackward0>)

In [3]:
from core.bias import RidgeBias, BiasWithCrossEntropy, BiasWithBCE
module = BiasWithBCE(
        predictive_model=noisy_model,
        bias_model=RidgeBias(),
        grad_match_loss_fn=nn.functional.mse_loss,
        optimizer_cls=torch.optim.Adam,
        lr=1e-3,
    )

# Train using the Trainer interface.
trainer = Trainer(
    max_epochs=10,
    logger=WandbLogger(project="inductive-bias", name="ridge-bias-noisy"),
    callbacks=[WandBCallback(), EarlyStopping(monitor="train/loss")],
    )
trainer.fit(module, mnist)

/Users/jrudoler/Library/Caches/pypoetry/virtualenvs/inductive-bias-XFXDbqEv-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'predictive_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['predictive_model'])`.
/Users/jrudoler/Library/Caches/pypoetry/virtualenvs/inductive-bias-XFXDbqEv-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'bias_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['bias_model'])`.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/jrudoler/Library/Caches/pypoetry/virtualenvs/inductive-bias-XFXDbqEv-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:68: You passed in a `v


  | Name             | Type      | Params | Mode 
-------------------------------------------------------
0 | predictive_model | NoisyMLP  | 150 K  | train
1 | bias_model       | RidgeBias | 1      | train
-------------------------------------------------------
150 K     Trainable params
0         Non-trainable params
150 K     Total params
0.601     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode
/Users/jrudoler/Library/Caches/pypoetry/virtualenvs/inductive-bias-XFXDbqEv-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

bias/beta,█▄▃▃▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆█████
train/loss,█▇▄▄▃▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇███
bias/beta,-0.0
epoch,4
train/loss,0.0
trainer/global_step,9349


In [ ]:
module = BiasWithCrossEntropy(
        predictive_model=noiseless_model,
        bias_model=RidgeBias(),
        loss_fn=nn.functional.mse_loss,
        optimizer_cls=torch.optim.Adam,
        lr=1e-3,
    )

# Train using the Trainer interface.
trainer = Trainer(
    max_epochs=10,
    logger=WandbLogger(project="inductive-bias", name="ridge-bias-noiseless"),
    callbacks=[WandBCallback(), EarlyStopping(monitor="train/loss")],
    )
trainer.fit(module, mnist)

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'predictive_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['predictive_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'bias_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['bias_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
wandb: Currently logged in as: jhrudoler (jhrudoler-penn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type      | Params | Mode 
-------------------------------------------------------
0 | predictive_model | NoisyMLP  | 150 K  | train
1 | bias_model       | RidgeBias | 1      | train
-------------------------------------------------------
150 K     Trainable params
0         Non-trainable params
150 K     Total params
0.601     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/torch/autograd/graph.py:823: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:180.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


bias/beta,█▆▄▃▃▂▂▁▁▁▂▁▁▂▁▁▁▁▁▁▁▂▁▂▁▁▁▁▁▁▁▂▂▁▂▁▂▁▁▁
epoch,▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▆▆▆▆▆▆▆▆▆███████████
train/loss,█▂▂▂▂▃▃▂▄▄▂▃▂▂▁▂▃▁▁▁▂█▃▂▂▂▂▂▄▂▂▄▃▄▂▂▃▂▃▂
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇▇███
bias/beta,0.3799
epoch,3
train/loss,25.0396
trainer/global_step,7499


In [15]:
module = BiasWithCrossEntropy(
    predictive_model=no_stopping_model,
    bias_model=RidgeBias(),
    loss_fn=nn.functional.mse_loss,
    optimizer_cls=torch.optim.Adam,
    lr=1e-3,
)
trainer = Trainer(
    max_epochs=10,
    logger=WandbLogger(project="inductive-bias", name="ridge-bias-no-stopping"),
    callbacks=[WandBCallback(), EarlyStopping(monitor="train/loss")],
    )
trainer.fit(module, mnist)

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'predictive_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['predictive_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'bias_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['bias_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type      | Params | Mode 
-------------------------------------------------------
0 | predictive_model | NoisyMLP  | 150 K  | train
1 | bias_model       | RidgeBias | 1      | train
-------------------------------------------------------
150 K     Trainable params
0         Non-trainable params
150 K     Total params
0.601     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

bias/beta,▁▂▃▅▅▆▆▇▇▇▇▇▆████▇▆▇███▇▇███▇▆▇████▇█▇█▆
epoch,▁▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▃▃▆▆▆▆▆▆▆▆▆████████
train/loss,▁█▁▁▁▃▂▃▃▂▂▂▁▂▁▄▁▄▂▁▂▂▅▃▂▂▃▁▂▂▂▁▁▁▂▂▄▁▁▄
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇█████
bias/beta,1.27503
epoch,3
train/loss,6461.88672
trainer/global_step,7499


In [4]:

def predictive_loss_grad(
        predictions: torch.Tensor, targets: torch.Tensor, loss_fn
    ) -> torch.Tensor:
        # Compute per-sample gradient of the loss function with respect to the model params
        loss = loss_fn(predictions, targets)
        # Compute the gradient of the loss with respect to the model predictions (dL/dy_hat)
        # because we're using chain rule.
        per_sample_grad = torch.autograd.grad(
            loss, predictions, retain_graph=True, create_graph=True
        )[0]
        return per_sample_grad

# unit tests

# Test the predictive_loss_grad function on mse loss
def test_predictive_loss_grad_mse():
    predictions = torch.tensor([[1.0], [2.0], [3.0]], requires_grad=True)
    targets = torch.tensor([[1.5], [2.5], [3.5]])
    loss_fn = nn.MSELoss(reduction="sum")
    
    # Compute the gradient
    grad = predictive_loss_grad(predictions, targets, loss_fn)
    
    # Expected gradient: 2 * (predictions - targets)
    expected_grad = 2 * (predictions - targets)
    
    assert torch.allclose(grad, expected_grad), f"Expected {expected_grad}, but got {grad}"

test_predictive_loss_grad_mse()

In [5]:
# load models
noisy_model = NoisyMLP(out_features=1)
noisy_state_dict = torch.load("saved_models/noisy-mlp.pt")
noisy_model.load_state_dict(noisy_state_dict)
noisy_model.eval()  # Set the model to evaluation mode

noiseless_model = NoisyMLP(out_features=1)
noiseless_state_dict = torch.load("saved_models/noiseless-mlp.pt")
noiseless_model.load_state_dict(noiseless_state_dict)
noiseless_model.eval()  # Set the model to evaluation mode



NoisyMLP(
  (loss_func): BCEWithLogitsLoss()
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): Dropout(p=0.2, inplace=False)
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Dropout(p=0.2, inplace=False)
    (5): Linear(in_features=128, out_features=128, bias=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=128, out_features=128, bias=True)
    (9): ReLU()
    (10): Dropout(p=0.2, inplace=False)
    (11): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [7]:
test_x = torch.randn(2, 1, 28, 28)
noisy_pred = noisy_model(test_x)  # Test the model to ensure it's working
noiseless_pred = noiseless_model(test_x)  # Test the model to ensure it's working

In [8]:
noisy_pred

tensor([[-21.6854],
        [-20.3525]], grad_fn=<AddmmBackward0>)

In [9]:
noiseless_pred

tensor([[-14.5025],
        [-13.9359]], grad_fn=<AddmmBackward0>)

In [4]:
from core.data import WikiTextDataModule

In [5]:
wiki = WikiTextDataModule(batch_size=32)

In [9]:
wiki_train = wiki.train_dataloader()

In [18]:
wiki_train.dataset[0]

{'text': ''}

## Ali et al. Linear Regression

In [2]:
from core.models import LinearRegression
from core.data import FullBatchDataModule

# set seed ensure reproducibility
torch.manual_seed(56)
# deterministic behavior
torch.use_deterministic_algorithms(False)

p = 10 # input dimension / number of features
N = 1000 # number of samples
X = torch.randn(N, p)
# X = torch.diag(torch.randn(p))
betas = 3 * torch.randn(p)
y = X @ betas
dm = FullBatchDataModule(X, y)

eps = 1e-2
linear_model = LinearRegression(input_dim=p, output_dim=1, lr=eps/2, fit_intercept=False, init_zeros=True)
wandb_logger = WandbLogger(
    project="inductive-bias", name="linear-regression", log_model=False
)

lm_trainer = Trainer(
    max_epochs=150,
    accumulate_grad_batches=1,
    log_every_n_steps=1,
    logger=wandb_logger,
    callbacks=[
        WandBCallback(),
        EarlyStopping(monitor="train/loss", patience=5, mode="min"),
    ],
    accelerator=device,
    devices=n_devices,
)
lm_trainer.fit(linear_model, dm)

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jrudoler/.cache/pypoetry/virtualenvs/inductive ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
You are using a CUDA device ('NVIDIA L40S') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: jhrudoler (jhrudoler-penn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type    | Params | Mode 
----------------------------------------------
0 | linear    | Linear  | 10     | train
1 | loss_func | MSELoss | 0      | train
----------------------------------------------
10        Trainable params
0         Non-trainable params
10        Total params
0.000     Total estimated model params size (MB)
2         Modules in train mode
0         Modules in eval mode
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=150` reached.


epoch,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇█████
train/loss,██▇▇▇▆▅▅▅▅▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇█████
epoch,149
train/loss,2.55173
trainer/global_step,149


In [3]:
k = lm_trainer.current_epoch
# Compute beta iterates up to k
beta_iterates = [torch.zeros(p)]  # Initialize with beta_0
for i in range(1, k + 1):
    beta_prev = beta_iterates[-1]
    # beta_next = beta_prev + 2 * eps * (X.T @ y - X.T @ X @ beta_prev) / N # depends on the learning rate
    beta_next = beta_prev + eps * (X.T @ y - X.T @ X @ beta_prev) / N
    beta_iterates.append(beta_next)
    # print(f"Beta at step {i}: {beta_iterates[-1]}")
    
model_beta = linear_model.linear.weight.detach()
print(f"Final beta iterate: {beta_iterates[-1]}")
print(f"Model beta: {model_beta}")
assert torch.allclose(beta_iterates[-1], model_beta, atol=1e-2), "Final beta does not match the model's weight"

Final beta iterate: tensor([ 0.6691,  0.2917,  0.1443,  0.0829, -0.8503, -1.0883,  2.4975,  4.7535,
        -0.3251, -0.2721])
Model beta: tensor([[ 0.6691,  0.2917,  0.1443,  0.0829, -0.8503, -1.0883,  2.4975,  4.7535,
         -0.3251, -0.2721]])


In [4]:
print(list(linear_model.parameters()))
print(betas)
assert torch.allclose(betas, linear_model.linear.weight[0], atol=1e-1)


[Parameter containing:
tensor([[ 0.6691,  0.2917,  0.1443,  0.0829, -0.8503, -1.0883,  2.4975,  4.7535,
         -0.3251, -0.2721]], requires_grad=True)]
tensor([ 0.8902,  0.3665,  0.2820,  0.3181, -1.2177, -1.5383,  3.1239,  6.0392,
        -0.4617, -0.4277])


AssertionError: 

In [5]:
import torch

def compute_Q_matrix(X: torch.Tensor, k: int, eps: float, jit = None) -> torch.Tensor:
    r"""
    Compute the Q matrix used for regularization based on the data matrix `X`, 
    an iteration parameter `k`, and a step size `eps`.

    This constructs Q via eigendecomposition of the normalized covariance matrix:

        A = (1/n) XᵀX

    Then defines:

        D = S @ [(I - εS)^(-k) - I]⁻¹

    where S is the diagonal matrix of eigenvalues and the final Q is:

        Q = V D Vᵀ

    Args:
        X (torch.Tensor): Data matrix of shape (n, p).
        k (int): Number of gradient steps (must be ≥ 1).
        eps (float): Learning rate or step size. Must be < 1 / λ_max.

    Returns:
        torch.Tensor: Symmetric positive-definite Q matrix of shape (p, p).
    """
    n = X.shape[0]
    A = X.T @ X / n
    # if necessary, add jitter to ensure numerical stability
    if jit is not None:
        A += jit*torch.eye(A.shape[0], dtype=A.dtype, device=A.device)

    eigenvals, eigenvecs = torch.linalg.eigh(A)

    max_lr = 1 / eigenvals.max()
    assert eps < max_lr, f"eps {eps} is larger than max lr {max_lr}"

    S = torch.diag(eigenvals)
    I = torch.eye(len(eigenvals), dtype=S.dtype, device=S.device)

    D = S @ torch.linalg.inv(torch.matrix_power(I - eps * S, -k) - I)

    if not torch.allclose(D, torch.diag(torch.diag(D))):
        print("Warning: D is not diagonal")

    Q = eigenvecs @ D @ eigenvecs.T
    return Q

k = lm_trainer.current_epoch
print(f"epoch: {k}")

Q = compute_Q_matrix(X, k, eps)
print(Q)

epoch: 150
tensor([[ 0.2950, -0.0044,  0.0015, -0.0052, -0.0028,  0.0043,  0.0087, -0.0012,
         -0.0033,  0.0087],
        [-0.0044,  0.3135, -0.0105,  0.0066, -0.0108, -0.0075, -0.0030, -0.0053,
          0.0005, -0.0076],
        [ 0.0015, -0.0105,  0.2941,  0.0033, -0.0089, -0.0048, -0.0036,  0.0075,
          0.0019, -0.0096],
        [-0.0052,  0.0066,  0.0033,  0.2682, -0.0167, -0.0041, -0.0016,  0.0185,
          0.0023, -0.0169],
        [-0.0028, -0.0108, -0.0089, -0.0167,  0.2950,  0.0065, -0.0079, -0.0008,
         -0.0015,  0.0153],
        [ 0.0043, -0.0075, -0.0048, -0.0041,  0.0065,  0.2940,  0.0028, -0.0115,
          0.0086, -0.0073],
        [ 0.0087, -0.0030, -0.0036, -0.0016, -0.0079,  0.0028,  0.2687, -0.0043,
          0.0011, -0.0220],
        [-0.0012, -0.0053,  0.0075,  0.0185, -0.0008, -0.0115, -0.0043,  0.2752,
         -0.0027,  0.0073],
        [-0.0033,  0.0005,  0.0019,  0.0023, -0.0015,  0.0086,  0.0011, -0.0027,
          0.2860,  0.0016],
        

In [6]:
import torch

def compute_beta_closed_form(X: torch.Tensor, y: torch.Tensor, Q: torch.Tensor) -> torch.Tensor:
    r"""
    Compute closed-form solution minimizing (1/n) * ||y - Xβ||² + βᵀ Q β.
    

    .. math::
        \frac{1}{n}\,\|y - X\beta\|_2^2 \;+\; \beta^T\,Q\,\beta

    Solution:
    
    .. math::
        \beta_{\star} \;=\; \bigl(X^\top X + n\,Q\bigr)^{-1} \;X^\top\,y

    Args:
        X (torch.Tensor): Design matrix of shape (n, p)
        y (torch.Tensor): Target vector of shape (n,)
        Q (torch.Tensor): Penalty matrix of shape (p, p)

    Returns:
        torch.Tensor: Solution vector β* of shape (p,)
    """
    # Get the number of samples
    n = X.shape[0]

    # Form the matrix X^T X + nQ
    A = X.T @ X + n * Q  # shape: (p, p)

    # Right-hand side: X^T y
    b = X.T @ y  # shape: (p,)

    # Solve the linear system for beta
    beta_star = torch.linalg.solve(A, b)  # shape: (p,)

    return beta_star

beta_closed_form = compute_beta_closed_form(X, y, Q)
# Compare with model's beta
print(f"Closed-form solution: {str(beta_closed_form.detach().numpy())}")
print(f"Model beta: {str(model_beta[0].detach().numpy())}")

Closed-form solution: [ 0.6691412   0.29170787  0.14434351  0.08288036 -0.85028124 -1.088254
  2.497546    4.7534676  -0.3250918  -0.27211088]
Model beta: [ 0.66914254  0.2917044   0.14434326  0.08288135 -0.8502816  -1.088254
  2.4975438   4.7534723  -0.3250945  -0.27210534]


In [7]:
from core.estimators import BiasWithMSE, BiasWithAutodiffLoss
from core.bias import MatrixRidgeBias, DiagMatrixRidgeBias, RidgeBias
# reload BiasWithMSE even though it's already imported
# from importlib import reload
# from core import estimators, bias
# reload(estimators)
# reload(bias)

class PSDMatrixRidgeBias(nn.Module):
    """
    Learns a PSD matrix Q of shape (dim, dim) by
    parameterizing a lower-triangular matrix L and returning
    Q = L @ L.T.
    """
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        # We'll store the unconstrained lower-triangular entries
        # Initialize with small random values
        L_init = 0.01 * torch.randn(dim, dim)
        # We'll force it to be lower-triangular in the forward pass
        self.L_unconstrained = nn.Parameter(L_init, requires_grad=True)

    def forward(self, flattened_params: torch.Tensor) -> torch.Tensor:
        if flattened_params.shape != (self.dim,):
            raise ValueError(
                f"flattened_params should be of shape ({self.dim},), "
                f"but got {flattened_params.shape}"
            )
        # Create L as strictly lower-triangular or lower-triangular with diagonal
        L = torch.tril(self.L_unconstrained)
        # Then Q = L L^T is guaranteed symmetric PSD
        Q = L @ L.T
        loss = flattened_params @ Q @ flattened_params
        return loss

class DiagMatrixRidgeBias(nn.Module):
    def __init__(self, dim: int = 10, Q_init: torch.Tensor = None):
        super().__init__()
        self.dim = dim
        if Q_init is not None:
            if Q_init.shape != (dim,):
                raise ValueError(
                    f"Q_init should be of shape ({dim},), but got {Q_init.shape}"
                )
        self.Q = (
            nn.Parameter(Q_init, requires_grad=True)
            if Q_init is not None
            else nn.Parameter(torch.zeros(dim))
        )  # Initialize parameter

    def forward(self, flattened_params: torch.Tensor):
        """
        Compute the quadratic form x^T Q x, where x is the flattened parameters.
        This is equivalent to the L2 regularization term with a matrix Q.
        """
        # check that flattened_params is of shape (dim,)
        if flattened_params.shape != (self.dim,):
            raise ValueError(
                f"flattened_params should be of shape ({self.dim},), but got {flattened_params.shape}"
            )
        # Compute the quadratic form
        # Q_diag = nn.functional.softplus(self.Q)
        Q = torch.diag(self.Q)
        loss = flattened_params @ Q @ flattened_params
        return loss

torch.manual_seed(56)
# noisy_Q_init = Q.clone() + 0.1 * torch.randn(Q.shape[0], Q.shape[1])
# matrix_bias_model = MatrixRidgeBias(dim=p, Q_init=noisy_Q_init.clone())

matrix_bias_model = DiagMatrixRidgeBias(dim=p)
# matrix_bias_model = RidgeBias()
wandb_logger = WandbLogger(
    project="inductive-bias", name="matrix-ridge-bias", log_model=False
)

estimator = BiasWithMSE(
    predictive_model=linear_model,
    # predictive_loss_fn=torch.nn.functional.mse_loss,
    bias_model=matrix_bias_model,
    grad_match_loss_fn=torch.nn.functional.mse_loss,
    lr=1e-2,
    optimizer_cls=torch.optim.Adam,
)

trainer = Trainer(
    max_epochs=5000,
    accumulate_grad_batches=1,
    log_every_n_steps=1,
    logger=wandb_logger,
    callbacks=[
        WandBCallback(),
        EarlyStopping(monitor="train/loss", patience=150, mode="min"),
    ],
    accelerator=device,
    devices=n_devices,
)
trainer.fit(estimator, dm)

# Q_hat = torch.eye(p) * estimator.bias_model.beta.detach()
Q_hat = torch.diag(estimator.bias_model.Q.detach())

# minibatch_dataloader = torch.utils.data.DataLoader(TensorDataset(X, y), batch_size=N//2, shuffle=True)
# trainer.fit(estimator, train_dataloaders=minibatch_dataloader)

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'predictive_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['predictive_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'bias_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['bias_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                | Params | Mode 
-----------------------------------------------------------------
0 | predictive_model | LinearRegression    | 10     | train
1 | bias_model       | DiagMatrixRidgeBias | 10     | train
-----------------------------------------------------------------
20        Trainable params
0         Non-trainable params
20        Total params
0.000     Total estimated model params size (MB)
4         Modules in train mode
0         Modules in eval mode
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

bias/Q,▁▄▅▅▅▆▆▆▆▇▇▇▇███████████████████████████
bias/Q_grad,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
train/loss,█▆▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇█████
bias/Q,1.87472
bias/Q_grad,0.0
epoch,816
train/loss,0.0
trainer/global_step,816


In [8]:
# print("initial:\t", torch.diag(noisy_Q_init))
print("estimated:\n", Q_hat)
print("theoretical:\n", Q)

estimated:
 tensor([[0.3112, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.2533, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.5697, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 1.5386, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3444, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.3439, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.2660, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.2755, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.3480,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.3965]

In [9]:
print("estimated:", compute_beta_closed_form(X, y, Q_hat))
print("ground truth:", compute_beta_closed_form(X, y, Q))

estimated: tensor([ 0.6691,  0.2917,  0.1443,  0.0829, -0.8503, -1.0883,  2.4975,  4.7535,
        -0.3251, -0.2721])
ground truth: tensor([ 0.6691,  0.2917,  0.1443,  0.0829, -0.8503, -1.0883,  2.4975,  4.7535,
        -0.3251, -0.2721])


## Kernel Regression

In [16]:
# regression with RBF kernel
import torch
from torch import Tensor

from typing import Optional
import torch
from torch import Tensor

def rbf_kernel_torch(
    X: Tensor,
    Y: Optional[Tensor] = None,
    gamma: Optional[float] = None
) -> Tensor:
    """
    Compute the RBF (Gaussian) kernel between X and Y:
      K[i,j] = exp(-gamma * ||X[i] - Y[j]||^2)

    If Y is None, uses Y = X.
    If gamma is None, defaults to 1.0 / n_features.
    """
    if Y is None:
        Y = X
    # default gamma = 1/n_features
    if gamma is None:
        gamma = 1.0 / X.size(1)

    # ||x - y||^2 = ||x||^2 + ||y||^2 - 2 x·y
    # X_norm = (X**2).sum(dim=1, keepdim=True)      # (n_X, 1)
    # Y_norm = (Y**2).sum(dim=1, keepdim=True).t()  # (1, n_Y)
    # sq_dists = X_norm + Y_norm - 2.0 * X @ Y.t()  # (n_X, n_Y)
    sq_dists = torch.cdist(X, Y, p=2) ** 2  # (n_X, n_Y)
    return torch.exp(-gamma * sq_dists)

# from core.models import KernelRegression
from lightning.pytorch import LightningModule
from typing import Callable
from torch import Tensor

class KernelRegression(LightningModule):
    def __init__(
        self,
        kernel_function: Callable[[Tensor, Tensor], Tensor],
        n_train_samples: int,
        fit_intercept: bool = True,
        lr = 1e-2
    ):
        super().__init__()
        self.kernel_linear = nn.Linear(n_train_samples, 1, bias=fit_intercept)
        self.kernel_function = kernel_function
        self.loss_func = nn.MSELoss()
        self.save_hyperparameters()

    def forward(self, x):
        K_mat = self.kernel_function(x, x)
        return self.kernel_linear(K_mat)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        # make sure y is the right shape
        if len(y.shape) == 1:
            y = y.view(-1, 1).float()
        assert y.shape == y_hat.shape, f"y shape: {y.shape}, y_hat shape: {y_hat.shape}"
        loss = self.loss_func(y_hat, y)
        self.log("train/loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_func(y_hat, y)
        self.log("val/loss", loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.SGD(self.parameters(), lr=self.hparams.lr)

eps = 1e-2
rbf_kernel_reg = KernelRegression(
    kernel_function=rbf_kernel_torch, 
    n_train_samples=N, 
    fit_intercept=False, 
    lr=eps/2)
wandb_logger = WandbLogger(
    project="inductive-bias", name="kernel-regression", log_model=False
)

rbf_trainer = Trainer(
    max_epochs=150,
    accumulate_grad_batches=1,
    log_every_n_steps=1,
    logger=wandb_logger,
    callbacks=[
        WandBCallback(),
        EarlyStopping(monitor="train/loss", patience=150, mode="min"),
    ],
    accelerator=device,
    devices=n_devices,
)
rbf_trainer.fit(rbf_kernel_reg, dm)
k = rbf_trainer.current_epoch

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jrudoler/.cache/pypoetry/virtualenvs/inductive ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type    | Params | Mode 
--------------------------------------------------
0 | kernel_linear | Linear  | 1.0 K  | train
1 | loss_func     | MSELoss | 0      | train
--------------------------------------------------
1.0 K     Trainable params
0         Non-trainable params
1.0 K     Total params
0.004     Total estimated model params size (MB)
2         Modules in train mode
0         Modules in eval mode
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=150` reached.


epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▇▇▇▇▇▇███
train/loss,██▇▇▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
epoch,149
train/loss,8.07256
trainer/global_step,149


In [19]:
K = rbf_kernel_torch(X, X)
Q_kernel = compute_Q_matrix(K, k, eps, jit=1e-5)
print(Q_kernel)

tensor([[ 6.3684e-01, -2.5680e-03, -2.1651e-03,  ..., -2.2924e-05,
         -7.0588e-04,  1.1279e-03],
        [-2.5680e-03,  6.1124e-01, -3.3209e-03,  ...,  5.4819e-04,
          1.9942e-03, -6.1872e-03],
        [-2.1651e-03, -3.3209e-03,  5.9631e-01,  ..., -1.2166e-03,
         -4.5919e-03, -2.6204e-03],
        ...,
        [-2.2925e-05,  5.4819e-04, -1.2166e-03,  ...,  6.5065e-01,
          1.6073e-04, -1.6129e-03],
        [-7.0588e-04,  1.9942e-03, -4.5919e-03,  ...,  1.6073e-04,
          6.2223e-01,  1.6643e-03],
        [ 1.1279e-03, -6.1872e-03, -2.6204e-03,  ..., -1.6129e-03,
          1.6643e-03,  6.1320e-01]])


In [21]:
# Estimate the quadratic form using the estimated Q from matrix ridge bias
rbf_estimator = BiasWithMSE(
    predictive_model=rbf_kernel_reg,
    bias_model=DiagMatrixRidgeBias(dim=N),
    grad_match_loss_fn=torch.nn.functional.mse_loss,
    lr=1e-2,
    optimizer_cls=torch.optim.Adam,
)
wandb_logger = WandbLogger(
    project="inductive-bias", name="kernel-bias", log_model=False
)
rbf_trainer = Trainer(
    max_epochs=5000,
    accumulate_grad_batches=1,
    log_every_n_steps=1,
    logger=wandb_logger,
    callbacks=[
        WandBCallback(),
        EarlyStopping(monitor="train/loss", patience=150, mode="min"),
    ],
    accelerator=device,
    devices=n_devices,
)
kernel_dataloader = DataLoader(TensorDataset(X, y), batch_size=N, shuffle=True)
rbf_trainer.fit(rbf_estimator, kernel_dataloader)

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'predictive_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['predictive_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'bias_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['bias_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                | Params | Mode 
-----------------------------------------------------------------
0 | predictive_model | KernelRegression    | 1.0 K  | train
1 | bias_model       | DiagMatrixRidgeBias | 1.0 K  | train
-----------------------------------------------------------------
2.0 K     Trainable params
0         Non-trainable params
2.0 K     Total params
0.008     Total estimated model params size (MB)
4         Modules in train mode
0         Modules in eval mode
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

bias/Q,▁▁▁▂▂▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇█████
bias/Q_grad,▅▆▆▃▇▁▂▄▆▃▃▅▂▃▆▃▁▅▄▇▇▆▂▄▁█▃▄▁▁▂▃▂▅▂▅▄▄█▁
epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇████
train/loss,█▂▅▁▃▁▂▄▄▂▃▃▆▃▃▃▂▄▃▄▃▂▂▄▃▃▂▃▂▃▂▂▃▂▂▇▃▂▃▁
trainer/global_step,▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
bias/Q,9.52281
bias/Q_grad,0.00957
epoch,209
train/loss,0.2242
trainer/global_step,209


In [25]:
Q_hat_kernel = torch.diag(rbf_estimator.bias_model.Q.detach())
print("estimated Q from kernel bias:\n", Q_hat_kernel)
# now estimate the quadratic form using the estimated Q from kernel bias
print("theoretical Q from kernel regression:\n", Q_kernel)

estimated Q from kernel bias:
 tensor([[-0.4781,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000, -0.1918,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000, -0.1744,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ...,  0.3598,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000, -0.0770,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000, -0.3362]])
theoretical Q from kernel regression:
 tensor([[ 6.3684e-01, -2.5680e-03, -2.1651e-03,  ..., -2.2924e-05,
         -7.0588e-04,  1.1279e-03],
        [-2.5680e-03,  6.1124e-01, -3.3209e-03,  ...,  5.4819e-04,
          1.9942e-03, -6.1872e-03],
        [-2.1651e-03, -3.3209e-03,  5.9631e-01,  ..., -1.2166e-03,
         -4.5919e-03, -2.6204e-03],
        ...,
        [-2.2925e-05,  5.4819e-04, -1.2166e-03,  ...,  6.5065e-01,
          1.6073e-04, -1.6129e-03],
        [-7.0588e-04,  1.9942e-03, -4.5919e-03,  ...,  1.6073e-04,
    

In [ ]:
# Compare the estimated Q from kernel bias with the one from kernel regression
print("theoretical beta from Q bias:\n", compute_beta_closed_form(K, y, Q_kernel))
print("estimated beta from Q bias:\n", compute_beta_closed_form(K, y, Q_hat_kernel))

theoretical beta from Q bias:
 tensor([-1.0739e-01, -1.1045e-01, -2.7926e-01,  1.5101e-01,  1.5740e-01,
        -1.6971e-01,  8.9967e-02,  2.0490e-01, -6.9775e-02, -1.3175e-01,
         4.3075e-02, -1.1520e-01, -8.7130e-02,  2.0919e-01,  2.8632e-01,
         1.8549e-01,  1.9291e-01, -1.9897e-02,  1.3060e-02, -2.1602e-01,
        -3.2654e-01,  1.6414e-01, -1.4004e-02, -2.0919e-02,  3.0132e-02,
         1.8775e-01, -3.0773e-02,  4.0387e-01, -7.8967e-02, -1.8964e-02,
        -3.0412e-01, -5.0350e-02,  2.6735e-01,  1.9981e-01,  8.1528e-02,
        -7.0619e-02,  2.5038e-01, -6.6334e-02, -3.1272e-01,  3.8233e-01,
        -2.7885e-01,  4.1597e-02, -1.4884e-01, -1.5005e-01,  1.2651e-01,
        -5.4760e-02,  7.7702e-02, -6.4071e-02,  1.4277e-01, -2.0919e-01,
         2.1495e-01,  6.6637e-02,  1.5525e-01,  6.4792e-02,  3.3832e-01,
         1.8216e-02, -8.5815e-02, -1.6037e-01,  1.1218e-01,  1.0181e-01,
        -2.1572e-01, -3.4102e-01, -9.6583e-02,  1.4790e-01,  4.4174e-02,
         1.1462e-01,

### Extra

In [17]:
def is_psd(matrix: torch.Tensor, tol: float = 1e-8) -> bool:
    """Check if a symmetric matrix is positive semidefinite (PSD)."""
    if not torch.allclose(matrix, matrix.T, atol=tol):
        return False
    try:
        # Eigenvalues should be >= -tol for numerical stability
        eigvals = torch.linalg.eigvalsh(matrix)
        return torch.all(eigvals >= -tol).item()
    except RuntimeError:
        return False


def test_psd_identity():
    mat = torch.eye(3)
    assert is_psd(mat)

def test_psd_zero():
    mat = torch.zeros(4, 4)
    assert is_psd(mat)

def test_not_psd():
    mat = torch.tensor([[0.0, 1.0], [1.0, -3.0]])
    assert not is_psd(mat)

def test_non_symmetric():
    mat = torch.tensor([[1.0, 2.0], [0.0, 1.0]])
    assert not is_psd(mat)

def test_near_psd():
    mat = torch.tensor([[1.0, 0.99999999], [0.99999999, 1.0]])
    assert is_psd(mat)

# run all tests
test_psd_identity()
test_psd_zero()
test_not_psd()
test_non_symmetric()
test_near_psd()